In [ ]:
import pandas as pd
import os
import glob
import numpy as np
from scipy import stats
import warnings
import ast
from sklearn.neighbors import LocalOutlierFactor  # 导入LOF算法
import matplotlib.pyplot as plt

# 设置中文字体（避免可视化乱码）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

# ---------------------- 第一步：配置路径（请替换为你的实际路径） ----------------------
FOLDER_PATH_AIDAS = "workdir/aidascd4tdonorresultqc3"
FOLDER_PATH_EQTL = "workdir/eqtlcd4tdonorresultqc3"
META_AIDAS_MALE = "workdir/aidasmale_donor_age_results/male_donor_by_age_stage.xlsx"
META_AIDAS_FEMALE = "workdir/aidasfamale_donor_age_results/female_donor_by_age_stage.xlsx"
META_EQTL_MALE = "workdir/eqtlmale_donor_age_results/male_donor_by_age_stage.xlsx"
META_EQTL_FEMALE = "workdir/eqtlfamale_donor_age_results/female_donor_by_age_stage.xlsx"
OUTPUT_RAW_SUMMARY = "workdir/cd4traw_noise_汇总统计表格_完整版1.csv"
OUTPUT_FINAL_CORRECTED = "workdir/cd4tlof_corrected_noise.csv"
OUTPUT_LOF_PLOT = "workdir/lof_raw_noise_detection.png"  # LOF检测结果图
OUTPUT_DIFF_PLOT = "workdir/corrected_noise_diff_analysis.png"  # 差异分析图
# 新增：宽表和基线表输出路径
OUTPUT_GENE_WIDE_TABLE = "workdir/cd4t_gene_corrected_wide_table.csv"  # 基因水平校正宽表
OUTPUT_BASELINE_TABLE = "workdir/cd4t_baseline_summary_table.csv"  # 各项基线统计汇总表

def detect_outliers_by_lof(df, group_col='group', value_col='raw_noise', 
                           n_neighbors_ratio=0.05,  # 改为按比例（如5%）
                           min_neighbors=5, max_neighbors=50,  # 限制邻居数上下限
                           contamination=0.1):
    """
    改进版：按分组样本量的比例动态计算n_neighbors，适配非平衡分组
    :param n_neighbors_ratio: 邻居数占该组样本量的比例（如0.05=5%）
    :param min_neighbors: 最小邻居数（避免样本量极小时邻居数过少）
    :param max_neighbors: 最大邻居数（避免样本量极大时邻居数过多）
    """
    print("\n" + "="*60)
    print("🔍 开始LOF异常检测（按组动态计算邻居数，适配非平衡分组）")
    print("="*60)
    
    result_df = df.copy()
    result_df['is_outlier'] = np.nan
    result_df['lof_score'] = np.nan
    outlier_stats = {}
    
    groups = result_df[group_col].unique()
    for group in groups:
        group_df = result_df[result_df[group_col] == group].copy()
        valid_data = group_df.dropna(subset=[value_col])
        
        if len(valid_data) < min_neighbors:
            print(f"⚠️ {group}组有效样本数({len(valid_data)}) < 最小邻居数({min_neighbors})，跳过LOF检测")
            outlier_stats[group] = {
                'total_samples': len(group_df),
                'valid_samples': len(valid_data),
                'outlier_count': 0,
                'outlier_ratio': 0.0
            }
            continue
        
        # 动态计算该组的n_neighbors（按比例+上下限限制）
        group_n_neighbors = int(len(valid_data) * n_neighbors_ratio)
        group_n_neighbors = max(min_neighbors, min(group_n_neighbors, max_neighbors))
        print(f"\nℹ️ {group}组：样本量={len(valid_data)} → 动态计算邻居数={group_n_neighbors}（比例={n_neighbors_ratio}）")
        
        # 准备LOF输入数据
        X = valid_data[[value_col]].values
        # 初始化LOF模型（使用动态计算的邻居数）
        lof = LocalOutlierFactor(n_neighbors=group_n_neighbors, 
                                 contamination=contamination, 
                                 novelty=False)
        y_pred = lof.fit_predict(X)
        lof_scores = -lof.negative_outlier_factor_
        
        # 标记异常值
        valid_data.loc[:, 'is_outlier'] = (y_pred == -1)
        valid_data.loc[:, 'lof_score'] = lof_scores
        
        # 合并回原数据
        result_df.loc[valid_data.index, 'is_outlier'] = valid_data['is_outlier']
        result_df.loc[valid_data.index, 'lof_score'] = valid_data['lof_score']
        
        # 统计异常值信息
        outlier_count = valid_data['is_outlier'].sum()
        outlier_ratio = outlier_count / len(valid_data)
        outlier_stats[group] = {
            'total_samples': len(group_df),
            'valid_samples': len(valid_data),
            'n_neighbors_used': group_n_neighbors,  # 记录实际使用的邻居数
            'outlier_count': int(outlier_count),
            'outlier_ratio': float(outlier_ratio),
            'lof_score_mean': float(lof_scores.mean()),
            'lof_score_std': float(lof_scores.std()),
            'lof_score_threshold': float(np.percentile(lof_scores, 100*(1-contamination)))
        }
        
        print(f"\n📊 {group}组LOF检测结果：")
        print(f"  总样本数：{len(group_df)} | 有效样本数：{len(valid_data)}")
        print(f"  使用邻居数：{group_n_neighbors} | 异常样本数：{outlier_count} | 异常比例：{outlier_ratio:.2%}")
        print(f"  LOF得分均值±标准差：{lof_scores.mean():.4f}±{lof_scores.std():.4f}")
    
    # 全局统计
    total_valid = result_df.dropna(subset=[value_col, 'is_outlier'])
    global_outliers = total_valid['is_outlier'].sum()
    global_ratio = global_outliers / len(total_valid) if len(total_valid) > 0 else 0
    
    outlier_stats['global'] = {
        'total_samples': len(result_df),
        'valid_samples': len(total_valid),
        'outlier_count': int(global_outliers),
        'outlier_ratio': float(global_ratio)
    }
    
    print(f"\n📈 全局LOF检测统计：")
    print(f"  有效样本数：{len(total_valid)} | 异常样本数：{int(global_outliers)} | 异常比例：{global_ratio:.2%}")
    
    # 绘制LOF检测结果图
    plot_lof_results(result_df, value_col, group_col, OUTPUT_LOF_PLOT)
    
    return result_df, outlier_stats

def plot_lof_results(df, value_col, group_col, save_path):
    """绘制LOF检测结果可视化图"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 子图1：raw_noise分布（标记异常值）
    groups = df[group_col].unique()
    colors = ['#1f77b4', '#ff7f0e']
    for i, group in enumerate(groups):
        group_df = df[df[group_col] == group].dropna(subset=[value_col, 'is_outlier'])
        if len(group_df) == 0:
            continue
        
        # 正常样本
        normal = group_df[group_df['is_outlier'] == False]
        ax1.scatter(normal.index, normal[value_col], 
                   c=colors[i], alpha=0.6, label=f'{group}组-正常', s=30)
        # 异常样本
        outlier = group_df[group_df['is_outlier'] == True]
        ax1.scatter(outlier.index, outlier[value_col], 
                   c='red', alpha=0.8, label=f'{group}组-异常', s=60, marker='x')
    
    ax1.set_xlabel('样本索引')
    ax1.set_ylabel('raw_noise值')
    ax1.set_title('LOF异常检测结果 - raw_noise分布')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2：LOF得分分布
    for i, group in enumerate(groups):
        group_df = df[df[group_col] == group].dropna(subset=['lof_score'])
        if len(group_df) == 0:
            continue
        ax2.hist(group_df['lof_score'], bins=20, alpha=0.6, 
                label=f'{group}组', color=colors[i], edgecolor='black')
    
    # 添加异常值阈值线
    threshold = df.dropna(subset=['lof_score'])['lof_score'].quantile(0.9)
    ax2.axvline(x=threshold, color='red', linestyle='--', label=f'异常阈值(90%分位数)')
    ax2.set_xlabel('LOF得分（越大越异常）')
    ax2.set_ylabel('样本数')
    ax2.set_title('LOF得分分布')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ LOF检测结果图已保存至：{os.path.abspath(save_path)}")

# ---------------------- 新增：差异分析函数 ----------------------
def analyze_group_differences(df, group_col='group', value_col='corrected_noise', save_path=None):
    """
    分析两组校正后raw_noise的差异
    :param df: 包含校正后数据的DataFrame
    :param group_col: 分组列名
    :param value_col: 分析值列名
    :param save_path: 差异分析图保存路径
    :return: 差异分析结果字典
    """
    print("\n" + "="*60)
    print("📊 开始两组校正后raw_noise差异分析")
    print("="*60)
    
    # 准备数据
    groups = df[group_col].unique()
    if len(groups) != 2:
        print(f"⚠️  当前分组数({len(groups)})≠2，无法进行两组差异分析")
        return None
    
    group1, group2 = groups
    data1 = df[(df[group_col] == group1) & (df['is_outlier'] == False)][value_col].dropna()
    data2 = df[(df[group_col] == group2) & (df['is_outlier'] == False)][value_col].dropna()
    
    if len(data1) == 0 or len(data2) == 0:
        print(f"⚠️ {group1}组有效样本数：{len(data1)} | {group2}组有效样本数：{len(data2)}，无法分析")
        return None
    
    # 1. 描述性统计
    stats_result = {
        group1: {
            'count': len(data1),
            'mean': float(data1.mean()),
            'std': float(data1.std()),
            'median': float(data1.median()),
            'min': float(data1.min()),
            'max': float(data1.max()),
            'q25': float(data1.quantile(0.25)),
            'q75': float(data1.quantile(0.75))
        },
        group2: {
            'count': len(data2),
            'mean': float(data2.mean()),
            'std': float(data2.std()),
            'median': float(data2.median()),
            'min': float(data2.min()),
            'max': float(data2.max()),
            'q25': float(data2.quantile(0.25)),
            'q75': float(data2.quantile(0.75))
        }
    }
    
    # 2. 正态性检验（Shapiro-Wilk）
    shapiro1 = stats.shapiro(data1[:1000])  # 限制样本数避免计算超时
    shapiro2 = stats.shapiro(data2[:1000])
    stats_result['normality_test'] = {
        group1: {'statistic': float(shapiro1.statistic), 'pvalue': float(shapiro1.pvalue)},
        group2: {'statistic': float(shapiro2.statistic), 'pvalue': float(shapiro2.pvalue)}
    }
    
    # 3. 方差齐性检验（Levene）
    levene = stats.levene(data1, data2)
    stats_result['levene_test'] = {
        'statistic': float(levene.statistic),
        'pvalue': float(levene.pvalue)
    }
    
    # 4. 差异显著性检验
    if (shapiro1.pvalue > 0.05) and (shapiro2.pvalue > 0.05) and (levene.pvalue > 0.05):
        # 满足正态+方差齐性：独立样本t检验
        t_test = stats.ttest_ind(data1, data2, equal_var=True)
        test_type = '独立样本t检验'
    else:
        # 不满足：Mann-Whitney U检验
        u_test = stats.mannwhitneyu(data1, data2, alternative='two-sided')
        test_type = 'Mann-Whitney U检验'
        stats_result['difference_test'] = {
            'test_type': test_type,
            'statistic': float(u_test.statistic),
            'pvalue': float(u_test.pvalue),
            'significant': u_test.pvalue < 0.05
        }
    if test_type == '独立样本t检验':
        stats_result['difference_test'] = {
            'test_type': test_type,
            'statistic': float(t_test.statistic),
            'pvalue': float(t_test.pvalue),
            'significant': t_test.pvalue < 0.05
        }
    
    # 5. 效应量计算（Cohen's d 或 r）
    if test_type == '独立样本t检验':
        # Cohen's d
        pooled_std = np.sqrt(((len(data1)-1)*data1.std()**2 + (len(data2)-1)*data2.std()**2) / (len(data1)+len(data2)-2))
        cohens_d = (data1.mean() - data2.mean()) / pooled_std
        stats_result['effect_size'] = {
            'type': "Cohen's d",
            'value': float(cohens_d),
            'interpretation': '小效应' if abs(cohens_d) < 0.5 else '中等效应' if abs(cohens_d) < 0.8 else '大效应'
        }
    else:
        # r (Mann-Whitney U)
        z = stats_result['difference_test']['statistic'] / np.sqrt(len(data1)*len(data2))
        stats_result['effect_size'] = {
            'type': 'r (Mann-Whitney)',
            'value': float(z),
            'interpretation': '小效应' if abs(z) < 0.1 else '中等效应' if abs(z) < 0.3 else '大效应'
        }
    
    # 打印分析结果
    print(f"\n📋 描述性统计（仅非异常样本）：")
    for group in [group1, group2]:
        print(f"\n{group}组：")
        print(f"  样本数：{stats_result[group]['count']}")
        print(f"  均值±标准差：{stats_result[group]['mean']:.6f}±{stats_result[group]['std']:.6f}")
        print(f"  中位数：{stats_result[group]['median']:.6f}")
        print(f"  四分位数范围：{stats_result[group]['q25']:.6f}~{stats_result[group]['q75']:.6f}")
        print(f"  范围：{stats_result[group]['min']:.6f}~{stats_result[group]['max']:.6f}")
    
    print(f"\n🔬 统计检验结果：")
    print(f"  检验方法：{stats_result['difference_test']['test_type']}")
    print(f"  统计量：{stats_result['difference_test']['statistic']:.4f}")
    print(f"  p值：{stats_result['difference_test']['pvalue']:.6f}")
    print(f"  差异显著性：{'显著 (p<0.05)' if stats_result['difference_test']['significant'] else '不显著 (p≥0.05)'}")
    
    print(f"\n📏 效应量分析：")
    print(f"  效应量类型：{stats_result['effect_size']['type']}")
    print(f"  效应量值：{stats_result['effect_size']['value']:.4f}")
    print(f"  效应量解释：{stats_result['effect_size']['interpretation']}")
    
    # 6. 可视化差异分析结果
    if save_path:
        plot_difference_analysis(df, group_col, value_col, stats_result, save_path)
    
    return stats_result

def plot_difference_analysis(df, group_col, value_col, stats_result, save_path):
    """绘制差异分析可视化图"""
    groups = df[group_col].unique()
    group1, group2 = groups[:2]
    
    # 准备非异常样本数据
    data1 = df[(df[group_col] == group1) & (df['is_outlier'] == False)][value_col].dropna()
    data2 = df[(df[group_col] == group2) & (df['is_outlier'] == False)][value_col].dropna()
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 子图1：箱线图
    ax1.boxplot([data1, data2], labels=[group1, group2], patch_artist=True,
               boxprops=dict(facecolor='lightblue', alpha=0.7),
               medianprops=dict(color='red'))
    ax1.set_ylabel(value_col)
    ax1.set_title('两组校正后raw_noise箱线图（非异常样本）')
    ax1.grid(True, alpha=0.3)
    
    # 子图2：直方图
    ax2.hist(data1, bins=20, alpha=0.6, label=group1, color='#1f77b4', edgecolor='black')
    ax2.hist(data2, bins=20, alpha=0.6, label=group2, color='#ff7f0e', edgecolor='black')
    ax2.set_xlabel(value_col)
    ax2.set_ylabel('样本数')
    ax2.set_title('校正后raw_noise分布直方图')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3：核密度图
    kde1 = stats.gaussian_kde(data1)
    kde2 = stats.gaussian_kde(data2)
    x_range = np.linspace(min(data1.min(), data2.min()), max(data1.max(), data2.max()), 1000)
    ax3.plot(x_range, kde1(x_range), label=group1, color='#1f77b4', linewidth=2)
    ax3.plot(x_range, kde2(x_range), label=group2, color='#ff7f0e', linewidth=2)
    ax3.fill_between(x_range, kde1(x_range), alpha=0.2, color='#1f77b4')
    ax3.fill_between(x_range, kde2(x_range), alpha=0.2, color='#ff7f0e')
    ax3.set_xlabel(value_col)
    ax3.set_ylabel('密度')
    ax3.set_title('校正后raw_noise核密度曲线')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 子图4：均值±标准差条形图
    means = [stats_result[group1]['mean'], stats_result[group2]['mean']]
    stds = [stats_result[group1]['std'], stats_result[group2]['std']]
    x_pos = [0, 1]
    ax4.bar(x_pos, means, yerr=stds, capsize=10, 
           color=['#1f77b4', '#ff7f0e'], alpha=0.7, edgecolor='black')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels([group1, group2])
    ax4.set_ylabel(f'{value_col}（均值±标准差）')
    ax4.set_title('校正后raw_noise均值对比')
    # 添加显著性标记
    p_val = stats_result['difference_test']['pvalue']
    sig_label = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    ax4.text(0.5, max(means) + max(stds) * 1.2, sig_label, ha='center', fontsize=16)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # 添加统计信息文本
    test_info = f"检验方法：{stats_result['difference_test']['test_type']}\np值：{p_val:.6f}\n效应量：{stats_result['effect_size']['value']:.4f} ({stats_result['effect_size']['interpretation']})"
    fig.text(0.02, 0.02, test_info, fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ 差异分析图已保存至：{os.path.abspath(save_path)}")

# ---------------------- 原有函数（修改：强制使用g列作为基因名，其余逻辑完全不变） ----------------------
def process_single_donor(csv_path):
    """处理单个donor表格，保留原所有统计指标（含raw_noise和log2fitratio分布）
    修改：1. 强制使用g列提取基因名（确保基因名正确） 2. 总细胞数提取从n列改为m列 3. 拟合基因数从n列提取
    新增：1.保留所有基因的log2fitratio数据 2.计算log2fitratio前三个最大值的平均值（u）
    """
    try:
        # 1. 提取donorid
        file_name = os.path.basename(csv_path)
        donorid = file_name.replace("_CD4T.rds_gene_analysis.csv", "")
        if not donorid:
            print(f"警告：{csv_path} 无法提取donorid，跳过")
            return None
        
        # 2. 读取数据
        try:
            df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding='gbk', low_memory=False)
        
        # 基础校验：新增g列为必需列
        required_cols = ['fitratio', 'fdr', 'log2fitratio', 'cv2', 'u', 'g']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"警告：{donorid} 缺少关键列 {missing_cols}，跳过")
            return None
        if df.empty:
            print(f"警告：{donorid} 数据为空，跳过")
            return None
        
        # ========== 关键修改：强制使用g列作为基因名列，其余逻辑完全不变 ==========
        gene_col = 'g'  # 固定使用g列，不再自动识别其他列
        # 清洗基因名称：转为字符串、去除首尾空格、大写化（保证一致性，避免重复）
        df['gene_name_clean'] = df[gene_col].astype(str).str.strip().str.upper()
        
        # 去除空值和重复基因
        df_valid_genes = df.dropna(subset=['gene_name_clean', 'log2fitratio']).copy()
        df_valid_genes = df_valid_genes.drop_duplicates(subset=['gene_name_clean'])
        
        # 转换log2fitratio为数值型
        df_valid_genes['log2fitratio'] = pd.to_numeric(df_valid_genes['log2fitratio'], errors='coerce')
        # 过滤有效log2fitratio值
        df_valid_genes = df_valid_genes.dropna(subset=['log2fitratio'])
        
        # 保存基因log2fitratio数据（字典形式：{基因名: log2fitratio值}）
        gene_log2fitratio = dict(zip(df_valid_genes['gene_name_clean'], df_valid_genes['log2fitratio']))
        
        # 3. 核心指标提取（完全保留原有逻辑，无任何改动）
        total_genes = len(df)
        
        # 总细胞数（修改：从m列提取，第一个非空非0值）
        total_cells = np.nan
        if 'm' in df.columns:
            m_series = df['m'].dropna()
            if not m_series.empty:
                try:
                    m_numeric = pd.to_numeric(m_series, errors='coerce').dropna()
                    non_zero_m = m_numeric[m_numeric != 0]
                    if not non_zero_m.empty:
                        total_cells = int(non_zero_m.iloc[0])
                except Exception as e:
                    print(f"警告：{donorid} 提取总细胞数（m列）失败：{str(e)}")
        
        # adjusted R2（第一个非空非0值）
        adjusted_r2 = np.nan
        if 'Adjusted_R_squared' in df.columns:
            r2_series = df['Adjusted_R_squared'].dropna()
            if not r2_series.empty:
                try:
                    r2_numeric = pd.to_numeric(r2_series, errors='coerce').dropna()
                    non_zero_r2 = r2_numeric[r2_numeric != 0]
                    if not non_zero_r2.empty:
                        adjusted_r2 = non_zero_r2.iloc[0]
                except Exception as e:
                    print(f"警告：{donorid} 提取adjusted R2失败：{str(e)}")
        
        # 新增：拟合基因数（fittinggenecount）从n列提取
        fitting_gene_count = np.nan
        if 'n' in df.columns:
            n_series = df['n'].dropna()
            if not n_series.empty:
                try:
                    n_numeric = pd.to_numeric(n_series, errors='coerce').dropna()
                    non_zero_n = n_numeric[n_numeric != 0]
                    if not non_zero_n.empty:
                        fitting_gene_count = int(non_zero_n.iloc[0])
                except Exception as e:
                    print(f"警告：{donorid} 提取拟合基因数（n列）失败：{str(e)}")
        
        # 4. noise基因筛选（完全保留原有逻辑）
        df_valid = df.dropna(subset=['fitratio', 'fdr'])
        try:
            df_valid['fitratio'] = pd.to_numeric(df_valid['fitratio'], errors='coerce')
            df_valid['fdr'] = pd.to_numeric(df_valid['fdr'], errors='coerce')
            noise_genes = df_valid[(df_valid['fitratio'] > 1) & (df_valid['fdr'] < 0.05)].copy()
        except Exception as e:
            print(f"警告：{donorid} 筛选noise基因失败：{str(e)}")
            noise_genes = pd.DataFrame()
        noise_gene_count = len(noise_genes)
        
        # 5. 新增：log2fitratio>0.5和<0.5的noise基因数统计（完全保留原有逻辑）
        log2fitratio_gt1_count = 0  # log2fitratio>0.5的噪音基因数
        log2fitratio_lt1_count = 0  # log2fitratio<0.5的噪音基因数
        if noise_gene_count > 0:
            try:
                noise_genes['log2fitratio'] = pd.to_numeric(noise_genes['log2fitratio'], errors='coerce')
                log2fitratio_valid = noise_genes['log2fitratio'].dropna()
                log2fitratio_gt1_count = len(noise_genes[noise_genes['log2fitratio'] > 0.5].dropna(subset=['log2fitratio']))
                log2fitratio_lt1_count = len(noise_genes[noise_genes['log2fitratio'] < 0.5].dropna(subset=['log2fitratio']))
            except Exception as e:
                print(f"警告：{donorid} 统计log2fitratio分布失败：{str(e)}")
        
        # 6. raw_noise计算（完全保留原有逻辑，无任何改动）
        raw_noise = np.nan
        if noise_gene_count > 0:
            try:
                noise_genes['cv2'] = pd.to_numeric(noise_genes['cv2'], errors='coerce')
                valid_noise_genes = noise_genes[noise_genes['cv2'] < 10].dropna(subset=['cv2']).copy()
                valid_noise_count = len(valid_noise_genes)
                
                if valid_noise_count >= 500000:
                    raw_noise = valid_noise_genes.nsmallest(50, 'cv2')['cv2'].sum()
                else:
                    raw_noise = valid_noise_genes['log2fitratio'].sum() if valid_noise_count > 0 else np.nan
            except Exception as e:
                print(f"警告：{donorid} 计算raw_noise失败：{str(e)}")
        
        # 7. 其他原统计指标（完全保留原有逻辑，无任何改动）
        # 表达量统计
        expr_max = np.nan
        expr_min = np.nan
        if noise_gene_count > 0:
            try:
                noise_genes['u'] = pd.to_numeric(noise_genes['u'], errors='coerce')
                expr_max = noise_genes['u'].max()
                expr_min = noise_genes['u'].min()
            except Exception as e:
                print(f"警告：{donorid} 统计表达量失败：{str(e)}")
        
        # cv2统计
        cv2_max = np.nan
        cv2_min = np.nan
        cv2_gt_10_count = 0
        if noise_gene_count > 0:
            try:
                noise_genes['cv2'] = pd.to_numeric(noise_genes['cv2'], errors='coerce')
                cv2_max = noise_genes['cv2'].max()
                cv2_min = noise_genes['cv2'].min()
                cv2_gt_10_count = len(noise_genes[noise_genes['cv2'] > 10].dropna(subset=['cv2']))
            except Exception as e:
                print(f"警告：{donorid} 统计cv2失败：{str(e)}")
        
        # log2fitratio统计（平均值+分位数）
        log2fitratio_mean = np.nan
        log2fitratio_25 = np.nan
        log2fitratio_50 = np.nan
        log2fitratio_75 = np.nan
        # 新增：计算log2fitratio前三个最大值的平均值（u）
        log2fitratio_top3_avg = np.nan  # 记为u
        if noise_gene_count > 0:
            try:
                noise_genes['log2fitratio'] = pd.to_numeric(noise_genes['log2fitratio'], errors='coerce')
                log2fit_ratio_valid = noise_genes['log2fitratio'].dropna()
                if not log2fit_ratio_valid.empty:
                    log2fitratio_mean = log2fit_ratio_valid.mean()
                    log2fitratio_25 = log2fit_ratio_valid.quantile(0.25)
                    log2fitratio_50 = log2fit_ratio_valid.quantile(0.5)
                    log2fitratio_75 = log2fit_ratio_valid.quantile(0.75)
                    # 计算前三个最大值的平均值
                    top3_vals = log2fit_ratio_valid.nlargest(3)
                    if len(top3_vals) >= 1:  # 至少有一个值时计算平均（不足3个则取现有值的平均）
                        log2fitratio_top3_avg = top3_vals.mean()
            except Exception as e:
                print(f"警告：{donorid} 统计log2fitratio分位数或前三个最大值失败：{str(e)}")
        
        # 8. 整理结果（完全保留原有指标，新增基因数据）
        result = {
            'donorid': donorid,
            'raw_noise': round(raw_noise, 4) if pd.notna(raw_noise) else np.nan,  # 原"noise值"→明确为raw_noise
            'noise基因数': noise_gene_count,
            'log2fitratio>0.5的噪音基因数': log2fitratio_gt1_count,
            'log2fitratio<0.5的噪音基因数': log2fitratio_lt1_count,
            '总基因数': total_genes,
            '总细胞数': total_cells if pd.notna(total_cells) else np.nan,
            '拟合基因数(fitting_gene_count)': fitting_gene_count if pd.notna(fitting_gene_count) else np.nan,
            'adjusted R2': round(adjusted_r2, 4) if pd.notna(adjusted_r2) else np.nan,
            'noise基因表达量最大值': round(expr_max, 6) if pd.notna(expr_max) else np.nan,
            'noise基因表达量最小值': round(expr_min, 6) if pd.notna(expr_min) else np.nan,
            'cv2最大值': round(cv2_max, 4) if pd.notna(cv2_max) else np.nan,
            'cv2最小值': round(cv2_min, 4) if pd.notna(cv2_min) else np.nan,
            'cv2大于10的噪音基因数': cv2_gt_10_count,
            'log2fitratio平均值': round(log2fitratio_mean, 4) if pd.notna(log2fitratio_mean) else np.nan,
            'log2fitratio_25分位数': round(log2fitratio_25, 4) if pd.notna(log2fitratio_25) else np.nan,
            'log2fitratio_50分位数': round(log2fitratio_50, 4) if pd.notna(log2fitratio_50) else np.nan,
            'log2fitratio_75分位数': round(log2fitratio_75, 4) if pd.notna(log2fitratio_75) else np.nan,
            'log2fitratio_top3_avg': round(log2fitratio_top3_avg, 4) if pd.notna(log2fitratio_top3_avg) else np.nan,  # 新增u
            # 新增：保存基因log2fitratio数据（字典形式）
            'gene_log2fitratio': gene_log2fitratio,
            # 新增：记录基因总数
            '有效基因数': len(gene_log2fitratio)
        }
    
        return result
    
    except Exception as e:
        print(f"处理 {csv_path} 时出错：{str(e)}，跳过该文件")
        return None

def batch_process_folder(folder_path, group_name):
    """批量处理单个文件夹下所有目标CSV文件，添加组别标识"""
    csv_files = glob.glob(os.path.join(folder_path, "**", "*.rds_gene_analysis.csv"), recursive=True)
    
    if len(csv_files) == 0:
        print(f"错误：在 {folder_path} 中未找到任何 *.rds_gene_analysis.csv 文件")
        return None
    
    print(f"\n===== 开始处理 {group_name} 组 =====")
    print(f"共发现 {len(csv_files)} 个目标文件，开始批量处理...")
    results = []
    for idx, csv_file in enumerate(csv_files, 1):
        if idx % 10 == 0:
            print(f"已处理 {idx}/{len(csv_files)} 个文件")
        
        donor_result = process_single_donor(csv_file)
        if donor_result:
            donor_result['group'] = group_name  # 添加组别标识（aidas/eqtl）
            results.append(donor_result)
    
    final_df = pd.DataFrame(results)
    print(f"{group_name} 组处理完成！共成功提取 {len(final_df)} 个donor的指标")
    return final_df

def match_age_sex_ethnic(raw_summary_df):
    """
    核心适配：
    1. 解析元信息表中"字符串格式的列表"（如 "['A', 'B']" → ['A', 'B']）
    2. 去除所有特殊符号，精准匹配donor ID
    3. 分别适配aidas（JP_RIK_H133格式）和eqtl（58_58格式）
    """
    print("\n" + "="*60)
    print("📋 开始匹配年龄、性别元信息（4个表：aidas/eqtl各男女）")
    print("="*60)
    
    try:
        # 1. 检查原始汇总表的donor列和格式
        raw_donor_col = 'donorid'
        if raw_donor_col not in raw_summary_df.columns:
            print(f"❌ 原始汇总表缺少 {raw_donor_col} 列！无法匹配")
            return raw_summary_df
        
        # 打印原始汇总表donor示例
        raw_donors_sample = raw_summary_df[raw_donor_col].dropna().head(5).tolist()
        print(f"ℹ️  原始汇总表donor ID示例：{raw_donors_sample}")
        print(f"ℹ️  原始汇总表共有 {raw_summary_df[raw_donor_col].nunique()} 个唯一donor")
        
        # 原始donor标准化（去除空格、小写，用于匹配）
        raw_donors_normalized = {str(d).strip().lower(): d for d in raw_summary_df[raw_donor_col].dropna().unique()}
        
        # 2. 定义每个元信息表的配置
        meta_configs = [
            (META_AIDAS_MALE, 'aidas', 'male', 'male_donor_ids'),
            (META_AIDAS_FEMALE, 'aidas', 'female', 'female_donor_ids'),
            (META_EQTL_MALE, 'eqtl', 'male', 'male_donor_ids'),
            (META_EQTL_FEMALE, 'eqtl', 'female', 'female_donor_ids')
        ]
        
        meta_list = []
        for path, expected_group, expected_sex, donor_col in meta_configs:
            if not os.path.exists(path):
                print(f"⚠️  警告：{path} 文件不存在，跳过该表")
                continue
            
            # 读取元信息表
            if path.endswith('.xlsx') or path.endswith('.xls'):
                df = pd.read_excel(path)
            else:
                sep = '\t' if path.endswith('.tsv') else ','
                df = pd.read_csv(path, sep=sep)
            
            print(f"\n" + "-"*50)
            print(f"ℹ️  处理元信息表：{path}")
            print(f"   组别：{expected_group}，性别：{expected_sex}")
            
            # 检查必需列
            required_cols = [donor_col, 'age', 'development_stage']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"⚠️  缺少必需列：{missing_cols}，跳过")
                continue
            
            # 3. 解析"字符串格式的列表"（关键步骤！）
            expanded_rows = []
            for idx, row in df.iterrows():
                donor_str = str(row[donor_col]).strip()
                if not donor_str or donor_str == 'nan':
                    continue
                
                try:
                    # 解析字符串列表为真正的列表（如 "['A', 'B']" → ['A', 'B']）
                    donor_ids = ast.literal_eval(donor_str)
                    # 确保解析后是列表
                    if not isinstance(donor_ids, list):
                        donor_ids = [donor_ids]
                except:
                    # 解析失败时，降级为按逗号分割（兼容异常格式）
                    donor_ids = [d.strip() for d in donor_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '').split(',') if d.strip()]
                
                # 打印解析后的donor示例（每个表打印前2个）
                if idx == 0:
                    print(f"   解析后donor ID示例（前2个）：{donor_ids[:2]}")
                
                # 为每个donor生成一行数据
                for donor_id in donor_ids:
                    donor_id_clean = str(donor_id).strip()  # 清洗donor ID
                    if not donor_id_clean:
                        continue
                    expanded_rows.append({
                        'donorid': donor_id_clean,
                        'age': row['age'],
                        'development_stage': row['development_stage'],
                        'group': expected_group,
                        'sex': expected_sex
                    })
            
            # 转换为DataFrame并清洗
            df_expanded = pd.DataFrame(expanded_rows)
            if df_expanded.empty:
                print(f"⚠️  该表解析后无有效donor数据，跳过")
                continue
            
            # 标准化元信息表的donor ID（与原始表一致）
            df_expanded['donorid_normalized'] = df_expanded['donorid'].astype(str).str.strip().str.lower()
            df_expanded['age'] = pd.to_numeric(df_expanded['age'], errors='coerce')
            
            # 过滤：只保留原始汇总表中存在的donor，且年龄有效
            df_matched = df_expanded[
                df_expanded['donorid_normalized'].isin(raw_donors_normalized.keys()) &
                (df_expanded['age'] >= 0) & (df_expanded['age'] <= 120)
            ].drop_duplicates(subset=['donorid', 'group'])  # 去重（同一donor+组别只保留一条）
            
            # 打印该表的匹配结果
            print(f"   解析后总donor数：{len(df_expanded)}")
            print(f"   与原始表匹配的donor数：{len(df_matched)}")
            
            if len(df_matched) > 0:
                matched_sample = df_matched[['donorid', 'age', 'sex']].head(2)
                print(f"   匹配成功示例：\n{matched_sample}")
            
            # 恢复原始donor ID格式（避免修改用户数据）
            df_matched['donorid'] = df_matched['donorid_normalized'].map(raw_donors_normalized)
            meta_list.append(df_matched[['donorid', 'group', 'sex', 'age', 'development_stage']])
        
        # 4. 合并所有匹配成功的元信息
        if meta_list:
            meta_df = pd.concat(meta_list, ignore_index=True).drop_duplicates(subset=['donorid', 'group'])
            print(f"\n" + "-"*50)
            print(f"ℹ️  所有表合并后，共匹配到 {len(meta_df)} 个唯一donor（donorid+group）")
        else:
            meta_df = pd.DataFrame()
            print(f"\n⚠️  所有表均无匹配成功的donor")
        
        # 5. 与原始汇总表合并（左连接，保留所有原始donor）
        merged_df = pd.merge(
            raw_summary_df,
            meta_df,
            on=['donorid', 'group'],
            how='left'
        )
        
        # 6. 最终匹配统计
        total_donors = len(merged_df)
        matched_donors = merged_df['age'].notna().sum()
        unmatched_donors = total_donors - matched_donors
        
        print(f"\n" + "="*50)
        print(f"✅ 元信息匹配完成！")
        print(f"📊 最终统计：")
        print(f"   总donor数：{total_donors}")
        print(f"   成功匹配元信息的donor数：{matched_donors}（{matched_donors/total_donors*100:.1f}%）")
        print(f"   未匹配的donor数：{unmatched_donors}")
        
        # 打印匹配成功的示例
        if matched_donors > 0:
            matched_sample = merged_df[merged_df['age'].notna()][['donorid', 'group', 'sex', 'age']].head(3)
            print(f"\n🎉 匹配成功的donor示例：")
            print(matched_sample)
        
        return merged_df
    
    except Exception as e:
        print(f"\n❌ 匹配失败：{str(e)}")
        import traceback
        traceback.print_exc()
        return raw_summary_df

def balanced_nonlinear_mapping(delta_abs, max_delta=None, slope=15, center=0.4):
    """
    平衡版非线性映射：适度增强，避免过度矫正
    :param delta_abs: delta的绝对值（个体值 - 基线值）
    :param max_delta: 最大delta值（用于归一化，默认取所有delta的95分位数）
    :param slope: 非线性映射的斜率（越大矫正强度越高）
    :param center: 非线性映射的中心点（0-1之间）
    :return: 0-1区间的强度因子f（平衡版）
    """
    if max_delta is None or max_delta == 0:
        return 0
    # 平衡版sigmoid：可配置斜率和中心点
    normalized = delta_abs / max_delta
    f = 1 / (1 + np.exp(-slope * (normalized - center)))
    return np.clip(f, 0, 1)  # 确保在0-1之间

def calculate_balanced_correction_factor(delta_series, max_delta, threshold=0.2, reverse_logic=False, 
                                         slope=15, center=0.4, min_cf=0.3, max_cf=1.7):
    """
    平衡版校正因子计算：
    :param delta_series: delta值序列
    :param max_delta: 最大delta值
    :param threshold: 阈值（delta绝对值小于此值时不校正）
    :param reverse_logic: 是否反转校正逻辑
    :param slope: 非线性映射斜率（强度参数）
    :param center: 非线性映射中心点
    :param min_cf: 校正因子最小值
    :param max_cf: 校正因子最大值
    :return: 校正因子列表
    """
    correction_factors = []
    for delta in delta_series:
        if pd.isna(delta):
            correction_factors.append(np.nan)
            continue
        
        # 阈值控制：仅delta绝对值超过阈值时校正
        if abs(delta) <= threshold:
            correction_factors.append(1.0)
            continue
        
        # 计算平衡版强度因子f
        delta_abs = abs(delta)
        f = balanced_nonlinear_mapping(delta_abs, max_delta, slope=slope, center=center)
        
        # 校正逻辑（reverse_logic控制是否反转）
        if reverse_logic:
            # 反转逻辑：delta>0 → 1+f；delta<0 → 1-f
            a = 1 + f if delta > 0 else 1 - f
        else:
            # 原逻辑：delta>0 → 1-f；delta<0 → 1+f
            a = 1 - f if delta > 0 else 1 + f
        
        # 校正因子范围限制
        a = np.clip(a, min_cf, max_cf)
        correction_factors.append(a)
    return correction_factors

def calculate_corrected_noise(merged_df, 
                              # 细胞数校正因子参数
                              cell_slope=15, cell_center=0.4, cell_threshold=0, cell_min_cf=0.3, cell_max_cf=1.7,
                              # 总基因数/细胞数比值校正因子参数
                              gene_cell_slope=15, gene_cell_center=0.4, gene_cell_threshold=2, gene_cell_min_cf=0.3, gene_cell_max_cf=1.7,
                              # R2/拟合基因数比值校正因子参数
                              r2_fg_slope=10, r2_fg_center=0.5, r2_fg_threshold=0, r2_fg_min_cf=0.3, r2_fg_max_cf=1.7,
                              # Spearman相关系数阈值参数（新增）
                              spearman_min=0.1, spearman_max=0.9):
    """
    平衡版校正逻辑：完全保留原有逻辑，无任何改动
    1. 细胞数校正因子：平衡版映射+无阈值+0.2-1.8范围
    2. 总基因数/细胞数比值校正因子：平衡版映射+0.2阈值+0.2-1.8范围（替换原拟合基因数/细胞数）
    3. adjusted R2/fittinggenecount比值校正因子：保留反转逻辑，强度不变
    4. 新增：计算拟合基因数与raw_noise的Spearman相关系数（p1/aidas，p2/eqtl），并融入最终校正
    5. 新增：Spearman相关系数上下限阈值，超出则截断到阈值
    """
    print("\n" + "="*60)
    print("📈 开始计算平衡版校正后noise（适度增强，避免过度矫正）")
    print("="*60)
    
    # 仅使用非异常样本进行校正计算
    merged_df_clean = merged_df[merged_df['is_outlier'] == False].copy()
    print(f"\n⚠️  仅使用LOF清理后的非异常样本进行校正计算：")
    print(f"  原始样本数：{len(merged_df)}")
    print(f"  非异常样本数：{len(merged_df_clean)}")
    
    # 1. 拆分两组（aidas/eqtl）
    df_aidas = merged_df_clean[merged_df_clean['group'] == 'aidas'].copy()
    df_eqtl = merged_df_clean[merged_df_clean['group'] == 'eqtl'].copy()
    
    if len(df_aidas) == 0 or len(df_eqtl) == 0:
        print("❌ 至少一组无有效非异常数据，无法计算校正因子")
        merged_df['corrected_noise'] = np.nan
        merged_df['log2fitratio_top3_avg_norm'] = np.nan
        merged_df['cell_correction_factor'] = np.nan
        merged_df['gene_cell_ratio_correction_factor'] = np.nan  # 总基因数/细胞数比值因子（原fg_ratio）
        merged_df['r2_fg_ratio_correction_factor'] = np.nan  # adjusted R2/fittinggenecount比值因子
        merged_df['spearman_p1'] = np.nan  # aidas组拟合基因数与raw_noise的Spearman系数
        merged_df['spearman_p2'] = np.nan  # eqtl组拟合基因数与raw_noise的Spearman系数
        merged_df['spearman_p1_clipped'] = np.nan  # 截断后的p1
        merged_df['spearman_p2_clipped'] = np.nan  # 截断后的p2
        # 新增：总校正因子（用于基因水平校正）
        merged_df['total_correction_factor'] = np.nan
        return merged_df
    
    print(f"分组信息：aidas组 {len(df_aidas)} 个非异常donor，eqtl组 {len(df_eqtl)} 个非异常donor")
    
    # ========== 新增：计算拟合基因数与raw_noise的Spearman相关系数（带阈值截断） ==========
    print(f"\n【计算拟合基因数与raw_noise的Spearman相关系数（带上下限阈值）】")
    print(f"⚠️ Spearman系数阈值配置：最小值={spearman_min}，最大值={spearman_max}")
    # aidas组（p1）
    aidas_valid_corr = df_aidas.dropna(subset=['拟合基因数(fitting_gene_count)', 'raw_noise'])
    if len(aidas_valid_corr) >= 2:  # 至少2个样本才计算相关系数
        p1, p1_pvalue = stats.spearmanr(
            aidas_valid_corr['拟合基因数(fitting_gene_count)'],
            aidas_valid_corr['raw_noise']
        )
        # 确保系数在合理范围（避免极端值）
        p1 = np.clip(p1, -1, 1)
        # 应用上下限阈值截断
        p1_clipped = np.clip(p1, spearman_min, spearman_max)
    else:
        p1 = 1.0  # 样本不足时默认系数为1（不影响校正）
        p1_clipped = 1.0  # 截断后默认值
        p1_pvalue = np.nan
    
    # eqtl组（p2）
    eqtl_valid_corr = df_eqtl.dropna(subset=['拟合基因数(fitting_gene_count)', 'raw_noise'])
    if len(eqtl_valid_corr) >= 2:
        p2, p2_pvalue = stats.spearmanr(
            eqtl_valid_corr['拟合基因数(fitting_gene_count)'],
            eqtl_valid_corr['raw_noise']
        )
        p2 = np.clip(p2, -1, 1)
        # 应用上下限阈值截断
        p2_clipped = np.clip(p2, spearman_min, spearman_max)
    else:
        p2 = 1.0
        p2_clipped = 1.0
        p2_pvalue = np.nan
    
    # 打印相关系数结果（含截断前后对比）
    print(f"- aidas组（p1）：")
    print(f"  原始Spearman相关系数 = {p1:.4f}（p值 = {p1_pvalue:.4f}，有效样本数 = {len(aidas_valid_corr)}）")
    print(f"  截断后Spearman相关系数 = {p1_clipped:.4f}（阈值范围：{spearman_min}~{spearman_max}）")
    print(f"- eqtl组（p2）：")
    print(f"  原始Spearman相关系数 = {p2:.4f}（p值 = {p2_pvalue:.4f}，有效样本数 = {len(eqtl_valid_corr)}）")
    print(f"  截断后Spearman相关系数 = {p2_clipped:.4f}（阈值范围：{spearman_min}~{spearman_max}）")
    # =====================================================================
    
    # 2. 计算log2fitratio_top3_avg归一化（保留原有逻辑）
    aidas_u_mean = df_aidas['log2fitratio_top3_avg'].dropna().mean()
    eqtl_u_mean = df_eqtl['log2fitratio_top3_avg'].dropna().mean()
    print(f"\n【log2fitratio前三个最大值的平均值（u）统计】")
    print(f"- aidas组u的均值：{aidas_u_mean:.4f}（有效样本数：{df_aidas['log2fitratio_top3_avg'].notna().sum()}）")
    print(f"- eqtl组u的均值：{eqtl_u_mean:.4f}（有效样本数：{df_eqtl['log2fitratio_top3_avg'].notna().sum()}）")
    
    df_aidas['log2fitratio_top3_avg_norm'] = df_aidas['log2fitratio_top3_avg'] / aidas_u_mean if aidas_u_mean != 0 else np.nan
    df_eqtl['log2fitratio_top3_avg_norm'] = df_eqtl['log2fitratio_top3_avg'] / eqtl_u_mean if eqtl_u_mean != 0 else np.nan
    print(f"- aidas组u归一化完成（有效样本数：{df_aidas['log2fitratio_top3_avg_norm'].notna().sum()}）")
    print(f"- eqtl组u归一化完成（有效样本数：{df_eqtl['log2fitratio_top3_avg_norm'].notna().sum()}）")
    
    # 3. 平衡版细胞数校正因子计算
    print(f"\n【平衡版细胞数校正因子计算】")
    print(f"⚠️  配置：1.阈值={cell_threshold} 2.非线性映射斜率={cell_slope} 中心点={cell_center} 3.校正范围={cell_min_cf}-{cell_max_cf}")
    # 3.1 计算细胞数基线（所有有效细胞数的均值）
    all_valid_cells = pd.concat([
        df_aidas['总细胞数'].dropna(),
        df_eqtl['总细胞数'].dropna()
    ])
    baseline_cells = all_valid_cells.mean()
    print(f"- 所有有效细胞数的均值（基线）：{baseline_cells:.0f}")
    print(f"- 有效细胞数样本数：{len(all_valid_cells)}")
    
    # 3.2 计算每个个体的delta（个体细胞数 - 基线细胞数）
    df_aidas['cell_delta'] = df_aidas['总细胞数'] - baseline_cells
    df_eqtl['cell_delta'] = df_eqtl['总细胞数'] - baseline_cells
    
    # 3.3 计算max_delta（所有delta绝对值的95分位数）
    all_cell_delta_abs = pd.concat([
        df_aidas['cell_delta'].abs().dropna(),
        df_eqtl['cell_delta'].abs().dropna()
    ])
    max_cell_delta = all_cell_delta_abs.quantile(0.95) if len(all_cell_delta_abs) > 0 else 0
    print(f"- 所有细胞数delta绝对值的95分位数（max_delta）：{max_cell_delta:.0f}")
    
    # 3.4 计算平衡版细胞数校正因子
    df_aidas['cell_correction_factor'] = calculate_balanced_correction_factor(
        df_aidas['cell_delta'], max_cell_delta, 
        threshold=cell_threshold, reverse_logic=False,
        slope=cell_slope, center=cell_center,
        min_cf=cell_min_cf, max_cf=cell_max_cf
    )
    df_eqtl['cell_correction_factor'] = calculate_balanced_correction_factor(
        df_eqtl['cell_delta'], max_cell_delta, 
        threshold=cell_threshold, reverse_logic=False,
        slope=cell_slope, center=cell_center,
        min_cf=cell_min_cf, max_cf=cell_max_cf
    )
    
    # 打印细胞校正因子统计
    print(f"\n【平衡版个体细胞校正因子统计】")
    for group, df_group in [('aidas', df_aidas), ('eqtl', df_eqtl)]:
        valid_cf = df_group['cell_correction_factor'].dropna()
        if len(valid_cf) > 0:
            print(f"- {group}组：")
            print(f"  校正因子范围：{valid_cf.min():.4f} ~ {valid_cf.max():.4f}")
            print(f"  校正因子均值±标准差：{valid_cf.mean():.4f}±{valid_cf.std():.4f}")
            print(f"  有效样本数：{len(valid_cf)}")
            print(f"  偏离1的样本数：{len(valid_cf[valid_cf != 1.0])}")
    
    # 4. 平衡版总基因数/细胞数比值校正因子计算（替换原拟合基因数/细胞数）
    print(f"\n【平衡版总基因数/细胞数比值校正因子计算】")
    print(f"⚠️  配置：1.阈值={gene_cell_threshold} 2.非线性映射斜率={gene_cell_slope} 中心点={gene_cell_center} 3.校正范围={gene_cell_min_cf}-{gene_cell_max_cf}")
    # 4.1 计算两组的总基因数/细胞数比值均值
    # aidas组
    aidas_valid_gene_cell = df_aidas.dropna(subset=['总基因数', '总细胞数'])
    aidas_gene_cell_ratio_mean = (aidas_valid_gene_cell['总基因数'] / aidas_valid_gene_cell['总细胞数']).mean() if len(aidas_valid_gene_cell) > 0 else 0
    
    # eqtl组
    eqtl_valid_gene_cell = df_eqtl.dropna(subset=['总基因数', '总细胞数'])
    eqtl_gene_cell_ratio_mean = (eqtl_valid_gene_cell['总基因数'] / eqtl_valid_gene_cell['总细胞数']).mean() if len(eqtl_valid_gene_cell) > 0 else 0
    
    # 两组比值的均值作为基线
    baseline_gene_cell_ratio = (aidas_gene_cell_ratio_mean + eqtl_gene_cell_ratio_mean) / 2 if (aidas_gene_cell_ratio_mean + eqtl_gene_cell_ratio_mean) > 0 else 0
    print(f"- aidas组（总基因数/细胞数）比值均值：{aidas_gene_cell_ratio_mean:.6f}（有效样本数：{len(aidas_valid_gene_cell)}）")
    print(f"- eqtl组（总基因数/细胞数）比值均值：{eqtl_gene_cell_ratio_mean:.6f}（有效样本数：{len(eqtl_valid_gene_cell)}）")
    print(f"- 两组比值均值（基线）：{baseline_gene_cell_ratio:.6f}")
    print(f"- 修正阈值：基线±{gene_cell_threshold}")
    
    # 4.2 计算每个个体的总基因数/细胞数比值
    df_aidas['gene_cell_ratio'] = df_aidas['总基因数'] / df_aidas['总细胞数']
    df_eqtl['gene_cell_ratio'] = df_eqtl['总基因数'] / df_eqtl['总细胞数']
    
    # 4.3 计算个体delta（个体比值 - 基线比值）
    df_aidas['gene_cell_ratio_delta'] = df_aidas['gene_cell_ratio'] - baseline_gene_cell_ratio
    df_eqtl['gene_cell_ratio_delta'] = df_eqtl['gene_cell_ratio'] - baseline_gene_cell_ratio
    
    # 4.4 计算max_gene_cell_delta（所有delta绝对值的95分位数）
    all_gene_cell_delta_abs = pd.concat([
        df_aidas['gene_cell_ratio_delta'].abs().dropna(),
        df_eqtl['gene_cell_ratio_delta'].abs().dropna()
    ])
    max_gene_cell_delta = all_gene_cell_delta_abs.quantile(0.95) if len(all_gene_cell_delta_abs) > 0 else 0
    print(f"- 所有总基因数比值delta绝对值的95分位数（max_delta）：{max_gene_cell_delta:.6f}")
    
    # 4.5 计算平衡版总基因数比值校正因子
    df_aidas['gene_cell_ratio_correction_factor'] = calculate_balanced_correction_factor(
        df_aidas['gene_cell_ratio_delta'], max_gene_cell_delta, 
        threshold=gene_cell_threshold, reverse_logic=False,
        slope=gene_cell_slope, center=gene_cell_center,
        min_cf=gene_cell_min_cf, max_cf=gene_cell_max_cf
    )
    df_eqtl['gene_cell_ratio_correction_factor'] = calculate_balanced_correction_factor(
        df_eqtl['gene_cell_ratio_delta'], max_gene_cell_delta, 
        threshold=gene_cell_threshold, reverse_logic=False,
        slope=gene_cell_slope, center=gene_cell_center,
        min_cf=gene_cell_min_cf, max_cf=gene_cell_max_cf
    )
    
    # 打印总基因数比值校正因子统计
    print(f"\n【平衡版个体总基因数比值校正因子统计】")
    for group, df_group in [('aidas', df_aidas), ('eqtl', df_eqtl)]:
        valid_cf = df_group['gene_cell_ratio_correction_factor'].dropna()
        if len(valid_cf) > 0:
            # 统计修正/未修正数量
            uncorrected = len(valid_cf[valid_cf == 1.0])
            corrected = len(valid_cf) - uncorrected
            print(f"- {group}组：")
            print(f"  校正因子范围：{valid_cf.min():.4f} ~ {valid_cf.max():.4f}")
            print(f"  校正因子均值±标准差：{valid_cf.mean():.4f}±{valid_cf.std():.4f}")
            print(f"  有效样本数：{len(valid_cf)}（未修正：{uncorrected}，修正：{corrected}）")
    
    # 5. adjusted R2/fittinggenecount比值校正因子计算（保留反转逻辑，支持参数配置）
    print(f"\n【adjusted R2/fittinggenecount比值校正因子计算（反转逻辑）】")
    print(f"⚠️  配置：1.阈值={r2_fg_threshold} 2.非线性映射斜率={r2_fg_slope} 中心点={r2_fg_center} 3.校正范围={r2_fg_min_cf}-{r2_fg_max_cf}")
    # 5.1 计算两组的adjusted R2/fittinggenecount比值均值
    # aidas组
    aidas_valid_r2_fg = df_aidas.dropna(subset=['adjusted R2', '拟合基因数(fitting_gene_count)'])
    aidas_r2_fg_ratio_mean = (aidas_valid_r2_fg['adjusted R2'] / aidas_valid_r2_fg['拟合基因数(fitting_gene_count)']).mean() if len(aidas_valid_r2_fg) > 0 else 0
    
    # eqtl组
    eqtl_valid_r2_fg = df_eqtl.dropna(subset=['adjusted R2', '拟合基因数(fitting_gene_count)'])
    eqtl_r2_fg_ratio_mean = (eqtl_valid_r2_fg['adjusted R2'] / eqtl_valid_r2_fg['拟合基因数(fitting_gene_count)']).mean() if len(eqtl_valid_r2_fg) > 0 else 0
    
    # 两组比值的均值作为基线
    baseline_r2_fg_ratio = (aidas_r2_fg_ratio_mean + eqtl_r2_fg_ratio_mean) / 2 if (aidas_r2_fg_ratio_mean + eqtl_r2_fg_ratio_mean) > 0 else 0
    print(f"- aidas组（adjusted R2/fittinggenecount）比值均值：{aidas_r2_fg_ratio_mean:.8f}（有效样本数：{len(aidas_valid_r2_fg)}）")
    print(f"- eqtl组（adjusted R2/fittinggenecount）比值均值：{eqtl_r2_fg_ratio_mean:.8f}（有效样本数：{len(eqtl_valid_r2_fg)}）")
    print(f"- 两组比值均值（基线）：{baseline_r2_fg_ratio:.8f}")
    print(f"- 修正阈值：基线±{r2_fg_threshold}")
    
    # 5.2 计算每个个体的adjusted R2/fittinggenecount比值
    # 定义比值计算函数（处理缺失值和分母为0）
    def calc_r2_fg_ratio(row):
        adj_r2 = row['adjusted R2']
        fg_count = row['拟合基因数(fitting_gene_count)']
        if pd.isna(adj_r2) or pd.isna(fg_count) or fg_count == 0:
            return np.nan
        return adj_r2 / fg_count
    
    df_aidas['r2_fg_ratio'] = df_aidas.apply(calc_r2_fg_ratio, axis=1)
    df_eqtl['r2_fg_ratio'] = df_eqtl.apply(calc_r2_fg_ratio, axis=1)
    
    # 5.3 计算个体delta（个体比值 - 基线比值）
    df_aidas['r2_fg_ratio_delta'] = df_aidas['r2_fg_ratio'] - baseline_r2_fg_ratio
    df_eqtl['r2_fg_ratio_delta'] = df_eqtl['r2_fg_ratio'] - baseline_r2_fg_ratio
    
    # 5.4 计算max_r2_fg_delta（所有delta绝对值的95分位数）
    all_r2_fg_delta_abs = pd.concat([
        df_aidas['r2_fg_ratio_delta'].abs().dropna(),
        df_eqtl['r2_fg_ratio_delta'].abs().dropna()
    ])
    max_r2_fg_delta = all_r2_fg_delta_abs.quantile(0.95) if len(all_r2_fg_delta_abs) > 0 else 0
    print(f"- 所有R2/fg比值delta绝对值的95分位数（max_delta）：{max_r2_fg_delta:.8f}")
    
    # 5.5 计算反转逻辑的校正因子
    df_aidas['r2_fg_ratio_correction_factor'] = calculate_balanced_correction_factor(
        df_aidas['r2_fg_ratio_delta'], max_r2_fg_delta,
        threshold=r2_fg_threshold, reverse_logic=True,  # 启用反转逻辑
        slope=r2_fg_slope, center=r2_fg_center,
        min_cf=r2_fg_min_cf, max_cf=r2_fg_max_cf
    )
    df_eqtl['r2_fg_ratio_correction_factor'] = calculate_balanced_correction_factor(
        df_eqtl['r2_fg_ratio_delta'], max_r2_fg_delta,
        threshold=r2_fg_threshold, reverse_logic=True,  # 启用反转逻辑
        slope=r2_fg_slope, center=r2_fg_center,
        min_cf=r2_fg_min_cf, max_cf=r2_fg_max_cf
    )
    
    # 打印R2/fg比值校正因子统计
    print(f"\n【反转逻辑R2/fg比值校正因子统计】")
    for group, df_group in [('aidas', df_aidas), ('eqtl', df_eqtl)]:
        valid_cf = df_group['r2_fg_ratio_correction_factor'].dropna()
        if len(valid_cf) > 0:
            uncorrected = len(valid_cf[valid_cf == 1.0])
            corrected = len(valid_cf) - uncorrected
            print(f"- {group}组：")
            print(f"  校正因子范围：{valid_cf.min():.8f} ~ {valid_cf.max():.8f}")
            print(f"  校正因子均值±标准差：{valid_cf.mean():.8f}±{valid_cf.std():.8f}")
            print(f"  有效样本数：{len(valid_cf)}（未修正：{uncorrected}，修正：{corrected}）")
    
    # 6. 合并校正因子到原始数据框
    # 先合并aidas和eqtl的校正结果
    corrected_df = pd.concat([df_aidas, df_eqtl], ignore_index=True)
    
    # 只保留需要合并回原始数据的列
    merge_cols = [
        'donorid', 'group', 'log2fitratio_top3_avg_norm',
        'cell_correction_factor', 'gene_cell_ratio_correction_factor',
        'r2_fg_ratio_correction_factor', 'cell_delta', 'gene_cell_ratio_delta', 'r2_fg_ratio_delta'  # 新增delta列，用于后续基线表计算
    ]
    
    # 左连接回原始merged_df
    merged_df = pd.merge(
        merged_df,
        corrected_df[merge_cols],
        on=['donorid', 'group'],
        how='left'
    )
    
    # 7. 填充Spearman相关系数（按组填充）
    merged_df['spearman_p1'] = np.nan
    merged_df['spearman_p2'] = np.nan
    merged_df['spearman_p1_clipped'] = np.nan
    merged_df['spearman_p2_clipped'] = np.nan
    
    # aidas组填充p1，eqtl组填充p2
    merged_df.loc[merged_df['group'] == 'aidas', 'spearman_p1'] = p1
    merged_df.loc[merged_df['group'] == 'aidas', 'spearman_p1_clipped'] = p1_clipped
    merged_df.loc[merged_df['group'] == 'eqtl', 'spearman_p2'] = p2
    merged_df.loc[merged_df['group'] == 'eqtl', 'spearman_p2_clipped'] = p2_clipped
    
    # 8. 计算最终校正后的noise（融合所有校正因子+Spearman系数）
    print(f"\n【计算最终校正后的noise值】")
    # 初始化校正后noise列
    merged_df['corrected_noise'] = np.nan
    
    # 仅对非异常样本计算校正后noise
    valid_mask = (merged_df['is_outlier'] == False) & (merged_df['raw_noise'].notna())
    valid_df = merged_df[valid_mask].copy()
    
    if len(valid_df) > 0:
        # 计算总校正因子（所有校正因子相乘 + Spearman系数调整）
        valid_df['total_correction_factor'] = (
            valid_df['cell_correction_factor'].fillna(1.0) *
            valid_df['gene_cell_ratio_correction_factor'].fillna(1.0) *
            valid_df['r2_fg_ratio_correction_factor'].fillna(1.0)
        )
        
        # 按组应用Spearman系数调整
        valid_df.loc[valid_df['group'] == 'aidas', 'total_correction_factor'] *= valid_df['spearman_p1_clipped'].fillna(1.0)
        valid_df.loc[valid_df['group'] == 'eqtl', 'total_correction_factor'] *= valid_df['spearman_p2_clipped'].fillna(1.0)
        
        # 计算校正后noise（完全保留原有逻辑）
        valid_df['corrected_noise'] = valid_df['raw_noise'] * valid_df['total_correction_factor']
        
        # 合并回原始数据框
        merged_df.loc[valid_df.index, 'corrected_noise'] = valid_df['corrected_noise']
        merged_df.loc[valid_df.index, 'total_correction_factor'] = valid_df['total_correction_factor']
        
        # 打印校正结果统计
        print(f"- 非异常样本校正完成：有效样本数={len(valid_df)}")
        print(f"- 校正前raw_noise范围：{valid_df['raw_noise'].min():.4f} ~ {valid_df['raw_noise'].max():.4f}")
        print(f"- 校正后noise范围：{valid_df['corrected_noise'].min():.4f} ~ {valid_df['corrected_noise'].max():.4f}")
        print(f"- 总校正因子范围：{valid_df['total_correction_factor'].min():.4f} ~ {valid_df['total_correction_factor'].max():.4f}")
    else:
        print(f"- 无有效非异常样本，无法计算校正后noise")
    
    # 9. 异常样本的校正后noise设为NaN
    merged_df.loc[merged_df['is_outlier'] == True, 'corrected_noise'] = np.nan
    
    print(f"\n✅ 平衡版校正后noise计算完成！")
    print(f"- 原始raw_noise有效样本数：{merged_df['raw_noise'].notna().sum()}")
    print(f"- 校正后noise有效样本数：{merged_df['corrected_noise'].notna().sum()}")
    print(f"- 异常样本数（校正后设为NaN）：{merged_df['is_outlier'].sum()}")
    
    return merged_df

# ---------------------- 新增：生成基因水平校正宽表（保留原有校正逻辑） ----------------------
def build_gene_wide_table(final_df):
    """
    生成基因水平校正宽表：
    1. 提取所有唯一基因（来自g列）
    2. 应用个体总校正因子，对每个基因的log2fitratio进行校正
    3. 宽表格式：donorid+分组信息为行，基因为列，值为校正后log2fitratio
    """
    print("\n" + "="*60)
    print("📊 开始生成基因水平校正宽表")
    print("="*60)
    
    # 1. 筛选有效数据（非异常样本 + 有总校正因子 + 有基因log2fitratio数据）
    valid_donors = final_df[
        (final_df['is_outlier'] == False) &
        (final_df['total_correction_factor'].notna()) &
        (final_df['gene_log2fitratio'].notna())
    ].copy()
    
    if len(valid_donors) == 0:
        print("❌ 无有效捐赠者数据，无法生成宽表")
        return pd.DataFrame()
    
    print(f"ℹ️  有效捐赠者数（非异常+有校正因子+有基因数据）：{len(valid_donors)}")
    
    # 2. 提取所有唯一基因（合并所有捐赠者的基因列表）
    all_genes = set()
    for gene_dict in valid_donors['gene_log2fitratio']:
        if isinstance(gene_dict, dict):
            all_genes.update(gene_dict.keys())
    all_genes = sorted(list(all_genes))
    print(f"ℹ️  所有捐赠者合并后唯一基因数：{len(all_genes)}")
    
    # 3. 构建宽表数据
    wide_table_data = []
    for idx, row in valid_donors.iterrows():
        donor_info = {
            'donorid': row['donorid'],
            'group': row['group'],
            'sex': row['sex'] if pd.notna(row['sex']) else 'unknown',
            'age': row['age'] if pd.notna(row['age']) else np.nan,
            'total_correction_factor': row['total_correction_factor']
        }
        
        # 提取该捐赠者的基因log2fitratio数据
        gene_dict = row['gene_log2fitratio']
        correction_factor = row['total_correction_factor']
        
        # 对每个基因应用校正（保留原有逻辑：原始log2fitratio × 总校正因子）
        for gene in all_genes:
            raw_log2 = gene_dict.get(gene, np.nan)
            if pd.isna(raw_log2):
                donor_info[gene] = np.nan
                continue
            
            # 完全保留原有校正逻辑：不修改计算方式，仅应用总校正因子
            corrected_log2 = raw_log2 * correction_factor
            
            # 空值处理（与原有逻辑一致）
            if corrected_log2 == 0 or pd.isna(corrected_log2):
                donor_info[gene] = np.nan
            else:
                donor_info[gene] = round(corrected_log2, 6)
        
        wide_table_data.append(donor_info)
    
    # 4. 转换为DataFrame并清理
    wide_table = pd.DataFrame(wide_table_data)
    
    # 过滤全空基因列（无任何有效校正值的基因）
    gene_cols = [col for col in wide_table.columns if col in all_genes]
    non_empty_gene_cols = [col for col in gene_cols if wide_table[col].notna().sum() > 0]
    wide_table = wide_table[['donorid', 'group', 'sex', 'age', 'total_correction_factor'] + non_empty_gene_cols]
    
    print(f"✅ 宽表生成完成！")
    print(f"   - 宽表行数（捐赠者数）：{len(wide_table)}")
    print(f"   - 宽表列数（信息列+有效基因列）：{len(wide_table.columns)}")
    print(f"   - 有效基因列数（非全空）：{len(non_empty_gene_cols)}")
    
    return wide_table

def build_baseline_table(final_df):
    """
    生成基线统计汇总表（新增：三个矫正因子delta的95分位数）
    新增指标：
    1. cell_delta_95q：细胞数delta绝对值的95分位数
    2. gene_cell_ratio_delta_95q：总基因数/细胞数比值delta绝对值的95分位数
    3. r2_fg_ratio_delta_95q：R2/拟合基因数比值delta绝对值的95分位数
    """
    print("\n" + "="*60)
    print("📊 开始生成基线统计汇总表（含三个delta的95分位数）")
    print("="*60)
    
    # 1. 筛选非异常样本（与校正计算一致）
    clean_df = final_df[final_df['is_outlier'] == False].copy()
    if len(clean_df) == 0:
        print("❌ 无有效非异常样本，无法生成基线表")
        return pd.DataFrame()
    
    # 2. 拆分分组
    df_aidas = clean_df[clean_df['group'] == 'aidas'].copy()
    df_eqtl = clean_df[clean_df['group'] == 'eqtl'].copy()
    group_labels = ['aidas', 'eqtl']
    baseline_values = {}
    
    # 3. 计算细胞数基线（合并aidas/eqtl所有有效细胞数，取整体均值）
    all_valid_cells = pd.concat([
        df_aidas['总细胞数'].dropna(),
        df_eqtl['总细胞数'].dropna()
    ])
    baseline_cells = all_valid_cells.mean() if len(all_valid_cells) > 0 else np.nan
    baseline_values['baseline_cell_count'] = baseline_cells
    print(f"- 所有有效细胞数的均值（基线）：{baseline_cells:.0f}" if pd.notna(baseline_cells) else "- 所有有效细胞数的均值（基线）：无有效数据")
    print(f"- 有效细胞数样本数：{len(all_valid_cells)}")
    
    # 4. 计算基因/细胞比统一基线
    gene_cell_single_baselines = []
    for label in group_labels:
        group_df = clean_df[clean_df['group'] == label].copy()
        valid_gene_cell = group_df.dropna(subset=['总基因数', '总细胞数'])
        if len(valid_gene_cell) > 0:
            gene_cell_ratios = valid_gene_cell['总基因数'] / valid_gene_cell['总细胞数']
            single_group_gene_cell_baseline = gene_cell_ratios.mean()
            gene_cell_single_baselines.append(single_group_gene_cell_baseline)
    if len(gene_cell_single_baselines) == 2:
        baseline_values['baseline_gene_cell_ratio'] = (gene_cell_single_baselines[0] + gene_cell_single_baselines[1]) / 2
    elif len(gene_cell_single_baselines) == 1:
        baseline_values['baseline_gene_cell_ratio'] = gene_cell_single_baselines[0]
    else:
        baseline_values['baseline_gene_cell_ratio'] = np.nan
    
    # 5. 计算R2/fg比值统一基线
    r2_fg_single_baselines = []
    for label in group_labels:
        group_df = clean_df[clean_df['group'] == label].copy()
        valid_r2_fg = group_df.dropna(subset=['adjusted R2', '拟合基因数(fitting_gene_count)'])
        valid_r2_fg = valid_r2_fg[valid_r2_fg['拟合基因数(fitting_gene_count)'] > 0]
        if len(valid_r2_fg) > 0:
            r2_fg_ratios = valid_r2_fg['adjusted R2'] / valid_r2_fg['拟合基因数(fitting_gene_count)']
            single_group_r2_fg_baseline = r2_fg_ratios.mean()
            r2_fg_single_baselines.append(single_group_r2_fg_baseline)
    if len(r2_fg_single_baselines) == 2:
        baseline_values['baseline_r2_fg_ratio'] = (r2_fg_single_baselines[0] + r2_fg_single_baselines[1]) / 2
    elif len(r2_fg_single_baselines) == 1:
        baseline_values['baseline_r2_fg_ratio'] = r2_fg_single_baselines[0]
    else:
        baseline_values['baseline_r2_fg_ratio'] = np.nan
    
    # 6. 格式化统一基线值
    unified_cell_baseline = round(baseline_values['baseline_cell_count'], 0) if pd.notna(baseline_values['baseline_cell_count']) else np.nan
    unified_gene_cell_baseline = round(baseline_values['baseline_gene_cell_ratio'], 6) if pd.notna(baseline_values['baseline_gene_cell_ratio']) else np.nan
    unified_r2_fg_baseline = round(baseline_values['baseline_r2_fg_ratio'], 8) if pd.notna(baseline_values['baseline_r2_fg_ratio']) else np.nan
    
    # 7. 定义辅助函数：计算delta绝对值的95分位数
    def calc_delta_95q(df, delta_col):
        """计算指定delta列的绝对值95分位数"""
        if delta_col not in df.columns:
            return np.nan
        delta_abs = df[delta_col].abs().dropna()
        if len(delta_abs) == 0:
            return np.nan
        return np.quantile(delta_abs, 0.95)
    
    # 8. 构建分组数据（global/aidas/eqtl）
    groups_data = [
        ('global', clean_df),
        ('aidas', df_aidas),
        ('eqtl', df_eqtl)
    ]
    
    # 9. 构建基线数据（含新增的三个delta 95分位数）
    baseline_data = []
    for group_name, group_df in groups_data:
        group_stats = {'group': group_name}
        valid_count = len(group_df)
        group_stats['valid_sample_count'] = valid_count
        
        # 填充统一基线值
        group_stats['cell_baseline'] = unified_cell_baseline
        group_stats['gene_cell_ratio_baseline'] = unified_gene_cell_baseline
        group_stats['r2_fg_ratio_baseline'] = unified_r2_fg_baseline
        
        # 新增：计算三个delta的95分位数
        group_stats['cell_delta_95q'] = calc_delta_95q(group_df, 'cell_delta')
        group_stats['gene_cell_ratio_delta_95q'] = calc_delta_95q(group_df, 'gene_cell_ratio_delta')
        group_stats['r2_fg_ratio_delta_95q'] = calc_delta_95q(group_df, 'r2_fg_ratio_delta')
        
        if valid_count == 0:
            group_stats.update({
                'u_mean': np.nan,
                'spearman_correlation': np.nan,
                'spearman_clipped': np.nan
            })
            baseline_data.append(group_stats)
            continue
        
        # 10. 原有指标：u值均值
        u_mean = np.nan
        if 'log2fitratio_top3_avg' in group_df.columns:
            u_vals = group_df['log2fitratio_top3_avg'].dropna()
            u_mean = round(u_vals.mean(), 4) if len(u_vals) > 0 else np.nan
        group_stats['u_mean'] = u_mean
        
        # 11. 原有指标：Spearman相关系数
        spearman = np.nan
        spearman_clipped = np.nan
        if group_name == 'aidas':
            if 'spearman_p1' in group_df.columns and len(group_df['spearman_p1'].dropna()) > 0:
                spearman = group_df['spearman_p1'].dropna().unique()[0]
            if 'spearman_p1_clipped' in group_df.columns and len(group_df['spearman_p1_clipped'].dropna()) > 0:
                spearman_clipped = group_df['spearman_p1_clipped'].dropna().unique()[0]
        elif group_name == 'eqtl':
            if 'spearman_p2' in group_df.columns and len(group_df['spearman_p2'].dropna()) > 0:
                spearman = group_df['spearman_p2'].dropna().unique()[0]
            if 'spearman_p2_clipped' in group_df.columns and len(group_df['spearman_p2_clipped'].dropna()) > 0:
                spearman_clipped = group_df['spearman_p2_clipped'].dropna().unique()[0]
        
        group_stats['spearman_correlation'] = round(spearman, 4) if pd.notna(spearman) else np.nan
        group_stats['spearman_clipped'] = round(spearman_clipped, 4) if pd.notna(spearman_clipped) else np.nan
        
        # 格式化新增的95分位数（保持小数位数一致性）
        group_stats['cell_delta_95q'] = round(group_stats['cell_delta_95q'], 0) if pd.notna(group_stats['cell_delta_95q']) else np.nan
        group_stats['gene_cell_ratio_delta_95q'] = round(group_stats['gene_cell_ratio_delta_95q'], 6) if pd.notna(group_stats['gene_cell_ratio_delta_95q']) else np.nan
        group_stats['r2_fg_ratio_delta_95q'] = round(group_stats['r2_fg_ratio_delta_95q'], 8) if pd.notna(group_stats['r2_fg_ratio_delta_95q']) else np.nan
        
        baseline_data.append(group_stats)
    
    # 12. 转换为DataFrame并返回
    baseline_table = pd.DataFrame(baseline_data)
    
    # 打印最终结果
    print(f"\n✅ 基线表生成完成！")
    print(f"   - 基线表行数（分组数）：{len(baseline_table)}")
    print(f"   - 基线表列数（统计指标数）：{len(baseline_table.columns)}")
    print(f"\n📋 基线表预览（含三个delta 95分位数）：")
    print(baseline_table.round(4))
    
    return baseline_table

def main():
    """
    主函数：一键运行完整的分析流程（新增宽表和基线表生成）
    流程：
    1. 批量处理AIDAS和eQTL文件夹数据
    2. 合并原始数据并匹配年龄/性别元信息
    3. LOF异常检测（raw_noise）
    4. 计算平衡版校正后noise（含R2/fg比值校正）
    5. 两组差异分析（校正后noise）
    6. 生成基因宽表和基线表
    7. 保存所有结果文件
    """
    print("="*80)
    print("🎯 开始执行完整的noise分析流程（含宽表和基线表生成）")
    print("="*80)
    
    # ---------------------- 步骤1：批量处理文件夹数据 ----------------------
    print("\n📌 步骤1：批量处理AIDAS和eQTL文件夹")
    # 处理AIDAS组
    df_aidas_raw = batch_process_folder(FOLDER_PATH_AIDAS, group_name='aidas')
    # 处理eQTL组
    df_eqtl_raw = batch_process_folder(FOLDER_PATH_EQTL, group_name='eqtl')
    
    # 检查数据是否加载成功
    if df_aidas_raw is None or df_eqtl_raw is None:
        print("❌ 数据加载失败！请检查文件夹路径和文件格式")
        return
    
    # 合并原始数据
    df_raw_combined = pd.concat([df_aidas_raw, df_eqtl_raw], ignore_index=True)
    print(f"\n✅ 原始数据合并完成：共 {len(df_raw_combined)} 个donor（AIDAS:{len(df_aidas_raw)}, eQTL:{len(df_eqtl_raw)}）")
    
    # ---------------------- 步骤2：匹配年龄/性别元信息 ----------------------
    print("\n📌 步骤2：匹配年龄、性别、发育阶段元信息")
    df_merged = match_age_sex_ethnic(df_raw_combined)
    
    # ---------------------- 步骤3：LOF异常检测（raw_noise） ----------------------
    print("\n📌 步骤3：LOF算法检测raw_noise异常值")
    # 改进后（按比例动态计算）
    df_merged_with_outlier, outlier_stats = detect_outliers_by_lof(
        df_merged,
        group_col='group',
        value_col='raw_noise',
        n_neighbors_ratio=0.01,  # 每组取5%的样本作为邻居
        min_neighbors=1,        # 最小邻居数
        max_neighbors=50,        # 最大邻居数
        contamination=0.1
    )
    
    # ---------------------- 步骤4：计算平衡版校正后noise ----------------------
    print("\n📌 步骤4：计算平衡版校正后noise（含R2/fg比值校正）")
    df_final = calculate_corrected_noise(
        df_merged_with_outlier,
        # 细胞数校正参数
        cell_slope=15, cell_center=0.4, cell_threshold=0, cell_min_cf=0.3, cell_max_cf=1.7,
        # 总基因数/细胞数比值校正参数
        gene_cell_slope=15, gene_cell_center=0.4, gene_cell_threshold=2, gene_cell_min_cf=0.4, gene_cell_max_cf=1.6,
        # R2/fittinggenecount比值校正参数（反转逻辑）
        r2_fg_slope=10, r2_fg_center=0.5, r2_fg_threshold=0, r2_fg_min_cf=0.6, r2_fg_max_cf=1.4,
        # Spearman相关系数阈值
        spearman_min=0.4, spearman_max=1
    )
    
    # ---------------------- 步骤5：两组差异分析（校正后noise） ----------------------
    print("\n📌 步骤5：AIDAS vs eQTL 校正后noise差异分析")
    stats_result = analyze_group_differences(
        df_final,
        group_col='group',
        value_col='corrected_noise',
        save_path=OUTPUT_DIFF_PLOT
    )
    
    # ---------------------- 步骤6：生成基因宽表和基线表（新增） ----------------------
    print("\n📌 步骤6：生成基因水平校正宽表和基线统计汇总表")
    # 生成宽表
    gene_wide_table = build_gene_wide_table(df_final)
    # 生成基线表（含三个delta 95分位数）
    baseline_table = build_baseline_table(df_final)
    
    # ---------------------- 步骤7：保存所有结果文件 ----------------------
    print("\n📌 步骤7：保存所有分析结果文件")
    # 保存原始汇总表
    df_raw_combined.to_csv(OUTPUT_RAW_SUMMARY, index=False, encoding='utf-8-sig')
    print(f"✅ 原始汇总表已保存：{os.path.abspath(OUTPUT_RAW_SUMMARY)}")
    
    # 保存最终校正结果表
    df_final.to_csv(OUTPUT_FINAL_CORRECTED, index=False, encoding='utf-8-sig')
    print(f"✅ 最终校正结果表已保存：{os.path.abspath(OUTPUT_FINAL_CORRECTED)}")
    
    # 保存基因宽表
    if not gene_wide_table.empty:
        gene_wide_table.to_csv(OUTPUT_GENE_WIDE_TABLE, index=False, encoding='utf-8-sig')
        print(f"✅ 基因水平校正宽表已保存：{os.path.abspath(OUTPUT_GENE_WIDE_TABLE)}")
    
    # 保存基线表
    if not baseline_table.empty:
        baseline_table.to_csv(OUTPUT_BASELINE_TABLE, index=False, encoding='utf-8-sig')
        print(f"✅ 基线统计汇总表已保存：{os.path.abspath(OUTPUT_BASELINE_TABLE)}")
    
    # ---------------------- 流程结束 ----------------------
    print("\n" + "="*80)
    print("🎉 完整分析流程执行完成！")
    print(f"📊 关键结果：")
    print(f"   - 原始样本数：{len(df_raw_combined)}")
    print(f"   - 匹配元信息样本数：{df_merged['age'].notna().sum()}")
    print(f"   - 异常样本数：{df_merged_with_outlier['is_outlier'].sum()}")
    print(f"   - 校正后noise有效样本数：{df_final['corrected_noise'].notna().sum()}")
    if not gene_wide_table.empty:
        print(f"   - 基因宽表有效捐赠者数：{len(gene_wide_table)}")
        print(f"   - 基因宽表有效基因数：{len(gene_wide_table.columns) - 5}")
    print("="*80)

# ---------------------- 运行入口 ----------------------
if __name__ == "__main__":
    # 运行前检查必要路径
    print("🔍 运行前检查路径配置...")
    required_paths = [
        FOLDER_PATH_AIDAS,
        FOLDER_PATH_EQTL
    ]
    # 检查文件夹是否存在
    for path in required_paths:
        if not os.path.exists(path):
            print(f"❌ 错误：路径不存在 → {path}")
            print("请先修改代码顶部的路径配置，确保指向正确的文件夹！")
            exit(1)
    
    # 检查元信息表（非必需，仅提醒）
    meta_paths = [META_AIDAS_MALE, META_AIDAS_FEMALE, META_EQTL_MALE, META_EQTL_FEMALE]
    missing_meta = [p for p in meta_paths if not os.path.exists(p)]
    if missing_meta:
        print(f"⚠️  警告：以下元信息表不存在（非必需，仅影响年龄/性别匹配）：")
        for p in missing_meta:
            print(f"   - {p}")
    
    # 执行主函数
    main()